# TikTok 文字起こし・分析ツール

**使い方**
1. 上のメニュー「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU** を選んで保存（無料枠で使えないときは CPU のままでも動きますが遅くなります）
2. 「ランタイム」→「**すべてのセルを実行**」
3. 途中で Google ドライブへのアクセス許可を聞かれたら「許可」
4. 最後のセルでパスワードを決めて Enter → 表示される `https://xxxx.gradio.live` のリンクを開く
5. ユーザー名 `user`、決めたパスワードでログイン

データ（文字起こし・構成案・辞書）は Google ドライブの「マイドライブ / TikTok文字起こし」フォルダに保存されます。Mac には何も保存しません。
終わったら「ランタイム」→「ランタイムを接続解除して削除」で止めてください。

In [ ]:
# @title 1. 準備（ツールのダウンロードとインストール）
!nvidia-smi -L || echo "GPU なし：CPU で動かします（文字起こしが遅くなります）"
!rm -rf /content/TiTok- && git clone -q --depth 1 -b claude/beautiful-cori-0ern9t https://github.com/Daiki9615/TiTok-.git /content/TiTok-
%pip install -q -U -r /content/TiTok-/requirements.txt
print("準備完了")

In [ ]:
# @title 2. Google ドライブにつなぐ（保存場所）
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/TikTok文字起こし'  # @param {type:"string"}
import os; os.makedirs(DATA_DIR, exist_ok=True)
print("保存場所:", DATA_DIR)

In [ ]:
# @title 3. スプレッドシートにも一覧を書き出す（任意）
USE_SHEETS = True  # @param {type:"boolean"}
SHEET_NAME = 'TikTok文字起こしデータ'  # @param {type:"string"}
sheets = None
if USE_SHEETS:
    from google.colab import auth
    auth.authenticate_user()
    import google.auth, gspread, sys
    sys.path.insert(0, '/content/TiTok-')
    from tiktok_tool.storage import SheetsMirror
    creds, _ = google.auth.default()
    sheets = SheetsMirror(gspread.authorize(creds), SHEET_NAME)
    print("スプレッドシート連携: オン（取り込み・修正のたびに一覧を書き出します）")
else:
    print("スプレッドシート連携: オフ")

In [ ]:
# @title 4. 起動（パスワードを決めて Enter → 表示されたリンクを開く）
import sys, getpass
sys.path.insert(0, '/content/TiTok-')
from tiktok_tool.storage import DataStore
from tiktok_tool.fetcher import YtDlpFetcher
from tiktok_tool.asr import WhisperTranscriber
from tiktok_tool.app import launch

password = getpass.getpass('このツール用のパスワード（他人に教えない）: ')
transcriber = WhisperTranscriber()
print('音声認識:', transcriber.name, '（最初の 1 本はモデルのダウンロードで数分かかります）')
launch(DataStore(DATA_DIR), YtDlpFetcher(), transcriber, sheets, password=password, share=True, debug=True)